# eur_r2 — GRM panel QC

One-time step: filters the shared genome-wide panel to the round-2 keep list,
applies per-cohort MAF/HWE/missingness QC, and uploads the result as a
plink 1.9 BED file for the shard jobs.

**Run this before `06_grm_shards.ipynb`.**

Prerequisite: `02_round2_gate.ipynb` must have written `eur_r2_keep_ids.txt`.

## config

In [ ]:
import os, subprocess

WS_GS  = "gs://cloned-shared-env-pilot-wb-swift-sprout-7231/phenotypic_covariance_v9"
R_GS   = f"{WS_GS}/eur_r2"

# genome-wide unified panel (shared across all sample sets, no per-sample-set QC applied)
PANEL_GS = f"{WS_GS}/01_ancestry_filtering/genome_wide_panel/genome_wide_panel_v9"

KEEP_GS = f"{R_GS}/01_ancestry/round2/eur_r2_keep_ids.txt"

# output — used by 06_grm_shards.ipynb
GRM_INPUT_GS = f"{R_GS}/03_grm/grm_input"
BED_NAME     = "eur_r2_GRM_QC"

LOCAL = os.path.expanduser("~/scratch_eur_r2_grm_qc")
os.makedirs(LOCAL, exist_ok=True)

print(f"panel: {PANEL_GS}")
print(f"output: {GRM_INPUT_GS}/{BED_NAME}.bed")

## install plink2

In [ ]:
%%bash
BIN_DIR="$HOME/bin"; mkdir -p "$BIN_DIR"
if [ ! -x "$BIN_DIR/plink2" ]; then
  cd /tmp
  wget -q -O plink2.zip     "https://s3.amazonaws.com/plink2-assets/alpha7/plink2_linux_x86_64_20260504.zip"
  unzip -o -q plink2.zip plink2 -d "$BIN_DIR" && chmod +x "$BIN_DIR/plink2"
fi
export PATH="$HOME/bin:$PATH"
plink2 --version

## download genome-wide panel and keep list

The panel is large (~94 GB BED). Download only once; the cell skips if files
already exist locally.

In [ ]:
%%bash
set -eo pipefail

for ext in bed bim fam; do
  [ -f "${LOCAL}/gwpanel.$ext" ] ||     gcloud storage cp "${PANEL_GS}.$ext" "${LOCAL}/gwpanel.$ext"
  echo "$ext: $(du -sh "${LOCAL}/gwpanel.$ext" | cut -f1)"
done
gcloud storage cp "$KEEP_GS" "${LOCAL}/keep.txt"
echo "keep: $(wc -l < "${LOCAL}/keep.txt") participants" 

## QC in the round-2 cohort

`--keep` restricts to round-2 participants, so MAF and HWE are evaluated in
the analysis cohort, not the full genome-wide panel. `--make-bed` outputs
plink 1.9 format required by the shard computation.

In [ ]:
%%bash
set -eo pipefail
export PATH="$HOME/bin:$PATH"

plink2   --bfile "${LOCAL}/gwpanel"   --keep "${LOCAL}/keep.txt" --nonfounders   --maf 0.01 --hwe 1e-6 0.001 keep-fewhet --geno 0.05   --max-alleles 2   --threads $(nproc)   --make-bed --out "${LOCAL}/${BED_NAME}"

echo "samples: $(wc -l < "${LOCAL}/${BED_NAME}.fam")"
echo "variants: $(wc -l < "${LOCAL}/${BED_NAME}.bim")" 

## compute allele frequencies

Precomputed using plink 1.9 so the shard jobs can read them with
`--read-freq`, ensuring GRM normalization is consistent across shards
(each shard only sees a subset of samples).

In [ ]:
%%bash
set -eo pipefail

# plink 1.9 — try copying from another staged location; install if missing
PLINK1="$HOME/bin/plink"
if [ ! -x "$PLINK1" ]; then
  # copy from a previously staged location (avoids S3 download blocked by VPC-SC)
  gcloud storage cp "${WS_GS}/03_grm_shards/eur_D2/bin/plink" "$PLINK1" 2>/dev/null &&     chmod +x "$PLINK1" || {
    echo "plink 1.9 not found. Upload manually:"
    echo "  gcloud storage cp /path/to/plink1.9 ${WS_GS}/bin/plink1"
    exit 1
  }
fi

"$PLINK1"   --bfile "${LOCAL}/${BED_NAME}"   --freq --out "${LOCAL}/${BED_NAME}_freq"

echo "freq file: ${LOCAL}/${BED_NAME}_freq.frq" 

## upload

In [ ]:
%%bash
set -eo pipefail

for f in "${BED_NAME}.bed" "${BED_NAME}.bim" "${BED_NAME}.fam" "${BED_NAME}_freq.frq"; do
  gcloud storage cp "${LOCAL}/$f" "${GRM_INPUT_GS}/$f"
  echo "uploaded: $f"
done

## verify

In [ ]:
%%bash
for ext in bed bim fam; do
  gcloud storage ls -l "${GRM_INPUT_GS}/${BED_NAME}.$ext" 2>/dev/null     || echo "MISSING: ${BED_NAME}.$ext"
done
gcloud storage ls -l "${GRM_INPUT_GS}/${BED_NAME}_freq.frq" 2>/dev/null   || echo "MISSING: ${BED_NAME}_freq.frq"

echo "sample count:"
gcloud storage cat "${GRM_INPUT_GS}/${BED_NAME}.fam" | wc -l

# BED size in GB — update BED_SIZE_GB in 06_grm_shards.ipynb with this
echo "BED size:"
gcloud storage ls -l "${GRM_INPUT_GS}/${BED_NAME}.bed" | awk '{printf "%.1f GB\n", $1/1024/1024/1024}' 

Next: `05_phenotype_residualize.ipynb`, then `06_grm_shards.ipynb`.